In [203]:
#####All Packages#####

import math
import numpy as np,numpy.linalg
import pandas as pd
from scipy.linalg import cholesky
import random
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.cluster import OPTICS
from scipy.sparse.linalg import svds
from functools import partial
from sklearn.decomposition import NMF
import sys
import warnings
from fancyimpute import IterativeSVD

#####All functions#####

##to get a positive semidefinite correlation matrix
def _getAplus(A):
    eigval, eigvec = np.linalg.eig(A)
    Q = np.matrix(eigvec)
    xdiag = np.matrix(np.diag(np.maximum(eigval, 0)))
    return Q*xdiag*Q.T

def _getPs(A, W=None):
    W05 = np.matrix(W**.5)
    return  W05.I * _getAplus(W05 * A * W05) * W05.I

def _getPu(A, W=None):
    Aret = np.array(A.copy())
    Aret[W > 0] = np.array(W)[W > 0]
    return np.matrix(Aret)

def nearPD(A, nit=10):
    n = A.shape[0]
    W = np.identity(n) 
# W is the matrix used for the norm (assumed to be Identity matrix here)
# the algorithm should work for any diagonal W
    deltaS = 0
    Yk = A.copy()
    for k in range(nit):
        Rk = Yk - deltaS
        Xk = _getPs(Rk, W=W)
        deltaS = Xk - Rk
        Yk = _getPu(Xk, W=W)
    return Yk


##for the case we have division by zero
def division_0(n, d):
    return n / d if d else 0


#####K-Means Clustering#####
def k_means_clustering(clusters):
    
    
    #loop for the multiple imputation process
    for n in range(0,5):
        #I have to reset the dataframe df2_train, otherwise it would cumulate over all iterations
        df2_train = df2_train_reset
        #loop for the iterative imputation process
        for i in range(0,2):

            ##dftest: rows with i+1 missing values##
            dftest = df2_test_new[df2_test_new.isnull().sum(axis=1)==i+1]
            #tells us which cells have missing values
            missing = ~np.isfinite(dftest)
            #creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
            imputation = np.where(missing, 0, dftest)
            #clusters the complete data
            kmeans = KMeans(clusters, random_state=n).fit(df2_train)
            result = kmeans.cluster_centers_

            ##finding the best cluster##

            #array A contains all rows with i missing values
            A = np.asarray(dftest)
            #to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
            #to each cluster center
            list = []
            #loop: will go through all k rows with i+1 missing value
            #calculating the distance: I will calculate the distance without prior imputation as it is easier and 
            #the results are the same
            #C calculates the distance, calculations with missing values result in 'NaN'
            #D transforms 'NaN' in 0 and transforms the distance in absolute value
            #finally, the manhatten distance is calculated by summarizing each array
            for k in range(0,len(dftest)):
                C = A[k] - result
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            #list is containing n arrays with the manhattan distance to each cluster center
            #the lowest value will be determined and 
            #the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            #result contains all clusters of one set
            #result[E] contains an array which assigns each row one cluster 
            #E assigns each missing value the nearest cluster
            #result[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

            #previously, imputation[missing] was the array with all missing values 
            #in this step, this array is replaced by an array with the values of the cluster center
            imputation[missing] = result[E][missing]

            #now, imputation contains the rows with replaced values (as array)
            #creating a new dataframe from the results
            #from array to dataframe
            df2_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            ##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
            df2_train = pd.concat([df2_train, df2_test_new_imputed])

        #saving the dataset of the multiple imputation run n
        J = df2_train
        #after the 2nd run (n == 1): add the new dataset (=J) to the old one (=I, from the first run)
        #K is the cumulated dataset                         
        if n == 1:
            K = I + J
        #after the 3rd run +: add to the cumulated dataset of the previous run, the new dataset J     
        if n > 1:
            K = K + J
        #saving the dataset for the next run            
        I = df2_train        



    #the complete dataset is the result of 5 imputation runs, the datasets were added previously
    #now they are divided by 5
    estimated_df2 = K/5
    estimated_df2

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []

    #rescales the data for NMSE calculation
    df2_full_comp2 = df2_full_comp*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    estimated_df2 = estimated_df2*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)    
        
    #v: number of column
    for v in range(0,len(estimated_df2.columns)):
        #mean of the variable (for each column; mu in paper)
        mu = np.mean(df2_full_comp2[df2_full_comp2.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp2[df2_full_comp2.columns[v]] - estimated_df2[estimated_df2.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*abs(mu)
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_kmeans.append(NMSE)
    return (NMSE_values_kmeans)


def k_means_clustering2(clusters):
    NMSE_values_kmeans = []
    
    for n in range(0,5):
        df1_train = df1[df1.isnull().sum(axis=1)==0]
        for i in range(0,2):

            dftest = df1[df1.isnull().sum(axis=1)==i+1]
            missing = ~np.isfinite(dftest)
            imputation = np.where(missing, 0, dftest)
            kmeans = KMeans(clusters, random_state=n).fit(df1_train)
            result = kmeans.cluster_centers_

            A = np.asarray(dftest)
            list = []
            for k in range(0,len(dftest)):
                C = A[k] - result
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            imputation[missing] = result[E][missing]

            df1_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            df1_train = pd.concat([df1_train, df1_test_new_imputed])

        J = df1_train                       
        if n == 1:
            K = I + J
        if n > 1:
            K = K + J
        I = df1_train        

    estimated_df1 = K/5
    estimated_df1

    sqrterror = []

    df1_full_comp2 = df.loc[df1.index]*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    estimated_df1 = estimated_df1*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)    
        
    for v in range(0,len(estimated_df1.columns)):
        mu = np.mean(df1_full_comp2[df1_full_comp2.columns[v]])
        nominator = (df1_full_comp2[df1_full_comp2.columns[v]] - estimated_df1[estimated_df1.columns[v]])**2
        denominator = df1.isna().sum()[v]*abs(mu)
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_kmeans.append(NMSE)
    return (NMSE_values_kmeans)    



#####OPTICS Clustering#####
#OPTICS "finds core sample of high density and expands clusters from them"
#We iterate over the number of samples in a neighborhood for a point to be considered as a core point (fraction of sample size)
#example: 10%: a point will considered as high density point if at least 10% of the sample is in the neighborhood
#each sample gets a label for a cluster, except for noisy samples
def optics_clustering(distance):
    

    #loop for the multiple imputation process
    for n in range(0,5):
        #I have to reset the dataframe df2_train, otherwise it would cumulate over all iterations
        df2_train = df2_train_reset
        #loop for the iterative imputation process
        for i in range(0,2):
            ##dftest: rows with i+1 missing values##
            dftest = df2_test_new[df2_test_new.isnull().sum(axis=1)==i+1]
            #tells us which cells have missing values
            missing = ~np.isfinite(dftest)
            #creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
            imputation = np.where(missing, 0, dftest)
            
            #clusters the complete data
            clustering = OPTICS(min_samples = distance/1000, metric = 'manhattan').fit(df2_train)
            optics = clustering.labels_
            optics_clusters = []
            #iterate for each label
            for labels in range(0,np.max(optics)+1):
                #look for the samples with the same label
                a = np.where(optics==labels)
                core_array = np.asarray(df2_train)[a]
                #take the mean of the samples with the same label to get the center of the cluster
                b = np.sum(core_array, axis=0)/np.max(np.shape(a))
                #store the cluster center
                optics_clusters.append(b)

            ##finding the best cluster##

            #array A contains all rows with i missing values
            A = np.asarray(dftest)
            #array B contains all clusters of one subset 
            B = np.asarray(optics_clusters)
            #to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
            #to each cluster center
            list = []
            #loop: will go through all k rows with i+1 missing value
            #calculating the distance: I will calculate the distance without prior imputation as it is easier and 
            #the results are the same
            #C calculates the distance, calculations with missing values result in 'NaN'
            #D transforms 'NaN' in 0 and transforms the distance in absolute value
            #finally, the manhatten distance is calculated by summarizing each array
            for k in range(0,len(dftest)):
                C = A[k] - B
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            #list is containing n arrays with the manhattan distance to each cluster center
            #the lowest value will be determined and 
            #the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            #optics_clusters contains all clusters of one set
            #optics_clusters[E] contains an array which assigns each row one cluster 
            #E assigns each missing value the nearest cluster
            #optics_clusters[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

            #previously, imputation[missing] was the array with all missing values 
            #in this step, this array is replaced by an array with the values of the cluster center
            imputation[missing] = np.asarray(optics_clusters)[E][missing]

            #now, imputation contains the rows with replaced values (as array)
            #creating a new dataframe from the results
            #from array to dataframe
            df2_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            ##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
            df2_train = pd.concat([df2_train, df2_test_new_imputed])
                
        #saving the dataset of the multiple imputation run n
        J = df2_train
        #after the 2nd run (n == 1): add the new dataset (=J) to the old one (=I, from the first run)
        #K is the cumulated dataset                         
        if n == 1:
            K = I + J
        #after the 3rd run +: add to the cumulated dataset of the previous run, the new dataset J     
        if n > 1:
            K = K + J
        #saving the dataset for the next run            
        I = df2_train        



    #the complete dataset is the result of 5 imputation runs, the datasets were added previously
    #now they are divided by 5
    estimated_df2 = K/5
    estimated_df2

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []

    #rescales the data for NMSE calculation
    df2_full_comp2 = df2_full_comp*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    estimated_df22 = estimated_df2*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)        
    
    #v: number of column
    for v in range(0,len(estimated_df22.columns)):
        #mean of the variable before imputation (for each column; mu in paper)
        mu = np.mean(df2_full_comp2[df2_full_comp2.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp2[df2_full_comp2.columns[v]] - estimated_df22[estimated_df22.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*abs(mu)
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_optics.append(NMSE)
    return (NMSE_values_optics)


def optics_clustering2(distance):
    NMSE_values_optics = []
    #same as above but with the hold-out dataset df1    
    for n in range(0,5):
        df1_train = df1[df1.isnull().sum(axis=1)==0]
        for i in range(0,2):

            dftest = df1[df1.isnull().sum(axis=1)==i+1]
            missing = ~np.isfinite(dftest)
            imputation = np.where(missing, 0, dftest)
            clustering = OPTICS(min_samples = distance/1000, metric = 'manhattan').fit(df1_train)
            optics = clustering.labels_
            optics_clusters = []
            for labels in range(0,np.max(optics)+1):
                a = np.where(optics==labels)
                core_array = np.asarray(df1_train)[a]
                b = np.sum(core_array, axis=0)/np.max(np.shape(a))
                optics_clusters.append(b)

            A = np.asarray(dftest)
            B = np.asarray(optics_clusters)
            list = []
            for k in range(0,len(dftest)):
                C = A[k] - B
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            imputation[missing] = np.asarray(optics_clusters)[E][missing]

            df1_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            frames = [df1_train, df1_test_new_imputed]
            df1_train = pd.concat(frames)
                
        J = df1_train                      
        if n == 1:
            K = I + J
        if n > 1:
            K = K + J      
        I = df1_train        

    estimated_df1 = K/5

    sqrterror = []

    df1_full_comp2 = df.loc[df1.index]*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    estimated_df12 = estimated_df1*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)      
    
    for v in range(0,len(estimated_df12.columns)):
        mu = np.mean(df1_full_comp2[df1_full_comp2.columns[v]])
        nominator = (df1_full_comp2[df1_full_comp2.columns[v]] - estimated_df12[estimated_df12.columns[v]])**2
        denominator = df1[df1.isnull().sum(axis=1)!=0].isna().sum()[v]*abs(mu)
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_optics.append(NMSE)
    return (NMSE_values_optics)


#####matrix factorization SVD#####    
def NMF_run_iterated(rank):
    #initialize on complete data
    matrix = df2_train
    matrix1 = IterativeSVD(rank).fit_transform(np.asarray(matrix))
    est_matrix = pd.DataFrame(data = matrix1,
                            index = matrix.index,
                            columns = matrix.columns)    
    
    for n in range(1,3):        
        #iteration with n missing values
        dftest = df2_test_new[df2_test_new.isnull().sum(axis=1)==n]
        matrix = pd.concat([est_matrix,dftest])
        matrix1 = IterativeSVD(rank).fit_transform(matrix)
        est_matrix = pd.DataFrame(data = matrix1,
                                  index = matrix.index,
                            columns = matrix.columns)

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []
    NMSE_values_NMF = []
    #rescales the data for NMSE calculation
    df2_full_comp2 = df2_full_comp*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    est_matrix2 = est_matrix*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)            

    #v: number of column
    for v in range(0,len(est_matrix.columns)):
        #mean of the variable before imputation (for each column; mu in paper)
        mu = np.mean(df2_full_comp2[df2_full_comp2.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp2[df2_full_comp2.columns[v]] - est_matrix2[est_matrix2.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*mu
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_NMF.append(NMSE)
    return(NMSE_values_NMF)

def NMF_run_iterated2(rank):
    #for the hold-out dataset df1
    #initialize on complete data
    matrix = df1[df1.isnull().sum(axis=1)==0]
    est_matrix = pd.DataFrame(data = IterativeSVD(rank).fit_transform(matrix),
                            index = matrix.index,
                            columns = matrix.columns)    
    
    for n in range(1,3):        
        #iteration with n missing values
        dftest = df1[df1.isnull().sum(axis=1)==n]
        matrix = pd.concat([est_matrix,dftest])
        est_matrix = pd.DataFrame(data = IterativeSVD(rank).fit_transform(matrix),
                                  index = matrix.index,
                            columns = matrix.columns)

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []
    NMSE_values_NMF = []
    #rescales the data for NMSE calculation
    df1_full_comp2 = df.loc[df1.index]*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
    est_matrix2 = est_matrix*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)            

    #v: number of column
    for v in range(0,len(est_matrix.columns)):
        #mean of the variable before imputation (for each column; mu in paper)
        mu = np.mean(df1_full_comp2[df1_full_comp2.columns[v]])
        #splitting the NMSE calculation
        nominator = (df1_full_comp2[df1_full_comp2.columns[v]] - est_matrix2[est_matrix2.columns[v]])**2
        denominator = df1_test_new.isna().sum()[v]*mu
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_NMF.append(NMSE)
    return(NMSE_values_NMF)


In [169]:
df2_test_new

,roomMin,roomMax,priceMin,priceMax
96,-0.923694,-0.565002,NaN,-0.563876
158,-0.048859,-1.196717,NaN,-1.197183
476,-0.894686,-1.472048,NaN,-1.471373
738,-0.739048,NaN,NaN,-1.043504
757,-0.102061,NaN,NaN,-1.093036
...,...,...,...,...
19421,-1.383542,NaN,NaN,0.753896
19791,2.755843,2.050986,NaN,2.047791
19807,-1.177087,NaN,NaN,-0.697084
19825,-0.264338,NaN,-0.264088,0.304256


In [199]:
####Simulation of the dataset####

seed = 30
sample_size = 5000
#Correlations
r1_2 = 0.3
r1_3 = 0.6
r1_4 = 0.32
r2_3 = 0.2
r2_4 = 0.5
r3_4 = 0.4

corr_mat = np.array([[1.0, r1_2, r1_3, r1_4],
                    [r1_2, 1.0, r2_3, r2_4],
                    [r1_3, r2_3, 1.0, r3_4],
                    [r1_4, r2_4, r3_4, 1.0]])

#positive semidefinite correlation matrix
corr_mat1 = nearPD(corr_mat,nit=10)

#Compute the (upper) Cholesky decomposition matrix
upper_chol = cholesky(corr_mat1)

#Generate 4 series of normally distributed (Gaussian) numbers
np.random.seed(seed)
rnd1 = np.random.normal(20.0, 3.0, size=(sample_size, 2))
np.random.seed(seed)
rnd2 = np.random.normal(1000.0, 125.0, size=(sample_size, 2))
rnd = np.concatenate((rnd1, rnd2), axis=1)

#Compute the inner product of upper_chol and rnd
ans = rnd @ upper_chol
#scales the data
ans1 = (ans-np.mean(ans, axis=0))/np.sqrt(np.var(ans, axis=0))

#transform array in dataframe
df = pd.DataFrame(data = ans1,
            columns = ['roomMin', 'roomMax', 'priceMin', 'priceMax'])

#####Modifying the complete simulated dataset#####

#percentage missing
#a for roomMin, b for roomMax, c for priceMin, d for priceMax
a = 0.23
b = 0.61
c = 0.69
d = 0.21

#create the 'missingness matrix'
df_NaN = pd.DataFrame({'NaN1' : np.random.sample(len(df)),
                       'NaN2' : np.random.sample(len(df)),
                       'NaN3' : np.random.sample(len(df)),
                       'NaN4' : np.random.sample(len(df))})

#transform the random values (uniform dist from 0 to 1) into integers referring to the missingness fraction
df_NaN.loc[(df_NaN.NaN1 <= a), 'NaN1'] = 0
df_NaN.loc[(df_NaN.NaN1 > a), 'NaN1'] = 1

df_NaN.loc[(df_NaN.NaN2 <= b), 'NaN2'] = 0
df_NaN.loc[(df_NaN.NaN2 > b), 'NaN2'] = 1

df_NaN.loc[(df_NaN.NaN3 <= c), 'NaN3'] = 0
df_NaN.loc[(df_NaN.NaN3 > c), 'NaN3'] = 1

df_NaN.loc[(df_NaN.NaN4 <= d), 'NaN4'] = 0
df_NaN.loc[(df_NaN.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
df_with_missing = pd.DataFrame(df.values*df_NaN.values, columns=df.columns, index=df.index)

#remove rows where roomMin and roomMax or priceMin and priceMax are missing
Z = np.where((df_with_missing[df_with_missing.columns[0]] + df_with_missing[df_with_missing.columns[1]]==0) | (df_with_missing[df_with_missing.columns[2]] + df_with_missing[df_with_missing.columns[3]]==0))
df_new = df_with_missing.drop(Z[0])


#transform the 0 into NaN
df_new.loc[(df_new.roomMin == 0),'roomMin']= np.nan
df_new.loc[(df_new.roomMax == 0),'roomMax']= np.nan
df_new.loc[(df_new.priceMin == 0),'priceMin']= np.nan
df_new.loc[(df_new.priceMax == 0),'priceMax']= np.nan

#####Preparing data for performance testing and iterative imputation#####

np.random.seed(seed)
#splitting the dataset into two subsets
#df1 is for final performance testing (10% of the whole dataset) - the hold-out dataset
#df2 is for training and testing (90% of the whole dataset)
msk1 = np.random.rand(len(df_new)) < 0.1
df1 = df_new[msk1]
df2 = df_new[~msk1]

#removing all rows with missing values in df2 (D0 in paper)
df2_full = df2.dropna()
np.random.seed(seed)
#splitting into test (df2_test, D0,2 in paper) (20%) and training (df2_train, D0,1 in paper) (80%) set
msk2 = np.random.rand(len(df2_full)) < 0.8
df2_train = df2_full[msk2]
df2_test = df2_full[~msk2]
np.random.seed(seed)
#second missingness matrix to determine which values from the test dataset (D0,2) to remove 
#to have the same missingness pattern
df_NaN2 = pd.DataFrame({'NaN1' : np.random.sample(len(df2_test)),
                        'NaN2' : np.random.sample(len(df2_test)),
                        'NaN3' : np.random.sample(len(df2_test)),
                        'NaN4' : np.random.sample(len(df2_test))})

df_NaN2.loc[(df_NaN2.NaN1 <= a), 'NaN1'] = 0
df_NaN2.loc[(df_NaN2.NaN1 > a), 'NaN1'] = 1

df_NaN2.loc[(df_NaN2.NaN2 <= b), 'NaN2'] = 0
df_NaN2.loc[(df_NaN2.NaN2 > b), 'NaN2'] = 1

df_NaN2.loc[(df_NaN2.NaN3 <= c), 'NaN3'] = 0
df_NaN2.loc[(df_NaN2.NaN3 > c), 'NaN3'] = 1

df_NaN2.loc[(df_NaN2.NaN4 <= d), 'NaN4'] = 0
df_NaN2.loc[(df_NaN2.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
#df2_test_v1 (D0,2 mod in paper) is the test data set with the same missingness pattern as the original dataset (D in paper)
df2_test_v1 = pd.DataFrame(df2_test.values*df_NaN2.values, columns=df2_test.columns, index=df2_test.index)


#remove rows where roomMin and roomMax or priceMin and priceMax are missing and keeping the index
Y = df2_test_v1[(df2_test_v1[df2_test_v1.columns[0]] + df2_test_v1[df2_test_v1.columns[1]]==0) | (df2_test_v1[df2_test_v1.columns[2]] + df2_test_v1[df2_test_v1.columns[3]]==0)]
df2_test_new = df2_test_v1.drop(Y.index)

#transform the 0 into NaN
df2_test_new.loc[(df2_test_new.roomMin == 0),'roomMin']= np.nan
df2_test_new.loc[(df2_test_new.roomMax == 0),'roomMax']= np.nan
df2_test_new.loc[(df2_test_new.priceMin == 0),'priceMin']= np.nan
df2_test_new.loc[(df2_test_new.priceMax == 0),'priceMax']= np.nan

##at the beginning, there are already complete rows in the test set
##those rows have to added to the train set   

df2_test_new = df2_test_new[df2_test_new.isnull().sum(axis=1)!=0]
df2_train = pd.concat([df2_train,df2_test_new[df2_test_new.isnull().sum(axis=1)==0]])
df2_train_reset = pd.concat([df2_train,df2_test_new[df2_test_new.isnull().sum(axis=1)==0]])

#combination of df2_train and df2_test_new (for later performance testing)
df2_full1 = pd.concat([df2_train,df2_test_new])
df2_full_comp = df2_full.loc[df2_full1.index]



In [3]:
##executing k-means

if __name__ == '__main__':
    NMSE_values_kmeans = []
    NMSE_values_kmeans.append(Parallel(n_jobs=16)(delayed(k_means_clustering)(f) for f in range(1,40)))
    
    
##executing OPTICS 

if __name__ == '__main__':
    NMSE_values_optics = []
    NMSE_values_optics.append(Parallel(n_jobs=16)(delayed(optics_clustering)(g) for g in range(1,20)))
    
    
##executing NMF

NMF_run(7)

[1.897914315478298,
 2.1616610150810622,
 2.295542811044847,
 3.1400766526941157,
 3.2365637205031037,
 3.362954876916656]

In [186]:
##executing k-means

if __name__ == '__main__':
    NMSE_values_kmeans = []
    NMSE_values_kmeans.append(Parallel(n_jobs=16)(delayed(k_means_clustering)(f) for f in range(1,10)))
k_means_clustering2(NMSE_values_kmeans.index(min(NMSE_values_kmeans))+1)

[24.962747097270086]

In [187]:
NMSE_values_kmeans

[[[24.11453112012483],
  [20.955456889887465],
  [18.8970500620016],
  [17.319308492227936],
  [21.476901247688037],
  [16.45965800962886],
  [13.840882190436947],
  [15.432615358653932],
  [17.680981799771118]]]

In [93]:
import csv
with open("test.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Dataset","NMSE_hold-out","NMSE_train"])
    writer.writerow([Q,NMSE_values_kmeans])

11

In [206]:
import csv


with open("kmeans_results.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Dataset","NMSE_hold-out","NMSE_train"])
    for Q in range(0,3):    
        if __name__ == '__main__':
            NMSE_values_kmeans = []
            NMSE_values_kmeans.append(Parallel(n_jobs=16)(delayed(k_means_clustering)(f) for f in range(1,10)))
        writer.writerow([Q,k_means_clustering2(NMSE_values_kmeans[0].index(min(NMSE_values_kmeans[0]))+1),NMSE_values_kmeans[0]])
        
with open("optics_results.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Dataset","NMSE_hold-out","NMSE_train"])
    for Q in range(0,3):    
        if __name__ == '__main__':
            NMSE_values_optics = []
            NMSE_values_optics.append(Parallel(n_jobs=16)(delayed(optics_clustering)(g) for g in range(1,10)))
        writer.writerow([Q,optics_clustering2(NMSE_values_optics[0].index(min(NMSE_values_optics[0]))+1),NMSE_values_optics[0]])

        
with open("MF_results.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Dataset","NMSE_hold-out","NMSE_train"])
    for Q in range(0,3):
        NMSE_NMF_run = []
        for g in range(1,4):
            NMSE_NMF_run.append(NMF_run(g))
        writer.writerow([Q,NMF_run2(NMSE_NMF_run.index(min(NMSE_NMF_run))+1),NMSE_NMF_run])
                


In [211]:
NMSE_NMF_run[0]

[19.696002099361852]

In [224]:
from sklearn.impute import SimpleImputer
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
imp_mean.fit(df2_full_comp)
print(imp_mean.transform(df2_full_comp))

SimpleImputer(add_indicator=False, copy=True, fill_value=None,
              missing_values=nan, strategy='mean', verbose=0)

In [229]:
pd.DataFrame(data=imp_mean.transform(df1), index=df1.index, columns=df1.columns)

,roomMin,roomMax,priceMin,priceMax
22,-0.416116,0.049946,0.025859,0.868196
31,-0.428385,-1.147303,0.025859,-1.147202
36,-0.192247,0.049946,0.025859,-0.262210
63,0.736930,0.049946,0.737178,0.049931
73,-0.441489,0.049946,-0.441588,-0.294829
...,...,...,...,...
4875,0.441158,0.221101,0.025859,0.220532
4880,-1.335527,0.049946,0.025859,-0.251312
4923,-0.315424,-0.392458,-0.315610,-0.392164
4941,-0.721799,0.049946,-0.721372,0.444140


In [226]:
##calculating NMSE
#storing the NMSE value of each column and number of imputed values per observation
sqrterror = []
NMSE_values_NMF = []
#rescales the data for NMSE calculation
df1_full_comp2 = df.loc[df1.index]*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)
est_matrix2 = est_matrix*np.sqrt(np.var(ans, axis=0))+np.mean(ans, axis=0)            

#v: number of column
for v in range(0,len(est_matrix.columns)):
    #mean of the variable before imputation (for each column; mu in paper)
    mu = np.mean(df1_full_comp2[df1_full_comp2.columns[v]])
    #splitting the NMSE calculation
    nominator = (df1_full_comp2[df1_full_comp2.columns[v]] - est_matrix2[est_matrix2.columns[v]])**2
    denominator = df1_test_new.isna().sum()[v]*mu
    #for the case that we have 0 replaced values in one column
    F = division_0(nominator, denominator)
    sqrterror.append(np.sum(F))
NMSE = np.sum(sqrterror)
NMSE_values_NMF.append(NMSE)
return(NMSE_values_NMF)

,roomMin,roomMax,priceMin,priceMax
66,0.559324,-0.409976,0.558951,-0.411031
84,-0.422159,0.875828,-0.421517,0.876897
88,-0.684065,-0.197423,-0.684053,-0.196471
100,1.048800,0.785264,1.049088,0.784038
169,-0.568405,-1.032960,-0.568947,-1.032589
...,...,...,...,...
3814,1.103780,1.304093,1.104387,1.303033
3986,0.308995,-0.406058,0.308675,-0.406728
4138,0.973103,0.500742,0.973226,0.499493
4375,-1.149558,-2.007161,-1.150603,-2.006370


In [216]:
##executing NMF
NMSE_NMF_run = []
for g in range(1,4):
    NMSE_NMF_run.append(NMF_run(g))
NMF_run2(NMSE_NMF_run.index(min(NMSE_NMF_run))+1)
NMSE_NMF_run



[IterativeSVD] Iter 1: observed MAE=0.460683
[IterativeSVD] Iter 2: observed MAE=0.455983
[IterativeSVD] Iter 3: observed MAE=0.455548
[IterativeSVD] Iter 4: observed MAE=0.455368
[IterativeSVD] Iter 5: observed MAE=0.455290
[IterativeSVD] Iter 6: observed MAE=0.455251
[IterativeSVD] Iter 7: observed MAE=0.455234
[IterativeSVD] Iter 8: observed MAE=0.455226
[IterativeSVD] Iter 9: observed MAE=0.455223
[IterativeSVD] Iter 1: observed MAE=0.460683
[IterativeSVD] Iter 2: observed MAE=0.032336
[IterativeSVD] Iter 3: observed MAE=0.018850
[IterativeSVD] Iter 4: observed MAE=0.010529
[IterativeSVD] Iter 5: observed MAE=0.005838
[IterativeSVD] Iter 6: observed MAE=0.003235
[IterativeSVD] Iter 7: observed MAE=0.001794
[IterativeSVD] Iter 8: observed MAE=0.000996
[IterativeSVD] Iter 9: observed MAE=0.000553
[IterativeSVD] Iter 10: observed MAE=0.000307
[IterativeSVD] Iter 1: observed MAE=0.460683
[IterativeSVD] Iter 2: observed MAE=0.032336
[IterativeSVD] Iter 3: observed MAE=0.010454
[Iterativ

[[7.43510192595069], [0.4319460190467159], [0.596977287931243]]

In [214]:
NMSE_NMF_run.index(min(NMSE_NMF_run))

1